Celda 1: Importación de librerías y carga de datos

In [13]:
import pandas as pd
import numpy as np
import os
import time
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

warnings.filterwarnings('ignore')

PATH_MENSUAL = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\dataset\dataset_mensual.xlsx"
PATH_OUTPUT_DIR = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\resultados"
PATH_GRAFICAS = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\notebooks\prediction\graficas"

os.makedirs(PATH_OUTPUT_DIR, exist_ok=True)
os.makedirs(PATH_GRAFICAS, exist_ok=True)

df_mensual = pd.read_excel(PATH_MENSUAL)
if 'Periodo' in df_mensual.columns:
    df_mensual['Periodo'] = pd.to_datetime(df_mensual['Periodo'].astype(str))
    df_mensual = df_mensual.set_index('Periodo')

print(f"Dataset mensual: {len(df_mensual)} meses")
print(f"Columnas: {list(df_mensual.columns[:8])}")

Dataset mensual: 46 meses
Columnas: ['servicios_totales', 'monto_total', 'monto_promedio', 'monto_mediana', 'carroza_count', 'carroza_flores_count', 'auto_count', 'microbus_count']


Celda 2: Preparación de datos y split temporal

In [14]:
TARGETS = ['servicios_totales', 'monto_total']

total_meses = len(df_mensual)

if total_meses >= 36:
    test_size = 12
elif total_meses >= 18:
    test_size = 6
else:
    test_size = max(3, total_meses // 4)

train = df_mensual.iloc[:-test_size]
test  = df_mensual.iloc[-test_size:]

print(f"Total meses disponibles: {total_meses}")
print(f"Test size seleccionado: {test_size}")
print(f"Train: {len(train)} meses ({train.index[0].strftime('%Y-%m')} a {train.index[-1].strftime('%Y-%m')})")
print(f"Test:  {len(test)} meses ({test.index[0].strftime('%Y-%m')} a {test.index[-1].strftime('%Y-%m')})")
if len(train) < 12:
    print("ADVERTENCIA: menos de 12 meses de train. Los resultados pueden ser poco confiables.")

Total meses disponibles: 46
Test size seleccionado: 12
Train: 34 meses (2022-05 a 2025-02)
Test:  12 meses (2025-03 a 2026-02)


Celda 3: SARIMA

In [15]:
def entrenar_evaluar_sarima(y_train, y_test):
    start = time.time()
    exito = False
    if len(y_train) >= 24:
        try:
            model = SARIMAX(
                y_train,
                order=(1, 1, 1),
                seasonal_order=(1, 1, 0, 12),
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            results = model.fit(disp=False)
            exito = True
            print(f"  SARIMA usando estacionalidad anual (S=12)")
        except Exception as e:
            print(f"  SARIMA estacional falló ({e}), usando ARIMA simple.")

    if not exito:
        model = SARIMAX(
            y_train,
            order=(1, 1, 1),
            seasonal_order=(0, 0, 0, 0),
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        results = model.fit(disp=False)

    pred = results.forecast(steps=len(y_test))
    tiempo = time.time() - start
    mae  = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2   = r2_score(y_test, pred) if len(y_test) > 1 else float('nan')
    mape = np.mean(np.abs((y_test.values - pred.values) / np.where(y_test.values == 0, 1, y_test.values))) * 100
    return {
        'modelo': 'SARIMA',
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo_entrenamiento': tiempo,
        'predicciones': pred.tolist(), 'real': y_test.tolist()
    }

resultados = []
for target in TARGETS:
    y_train = train[target]
    y_test  = test[target]
    res = entrenar_evaluar_sarima(y_train, y_test)
    res['target'] = target
    resultados.append(res)
    print(f"SARIMA - {target}: MAE={res['MAE']:.2f}, R2={res['R2']:.4f}, Tiempo={res['tiempo_entrenamiento']:.2f}s")

  SARIMA usando estacionalidad anual (S=12)
SARIMA - servicios_totales: MAE=4.47, R2=-0.4287, Tiempo=0.11s
  SARIMA usando estacionalidad anual (S=12)
SARIMA - monto_total: MAE=160447.26, R2=-11.1371, Tiempo=0.06s


Celda 4: Prophet

In [16]:
def entrenar_evaluar_prophet(y_train, y_test):
    start = time.time()
    df_p = pd.DataFrame({'ds': y_train.index, 'y': y_train.values})

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10.0,
        seasonality_mode='additive'
    )
    if len(y_train) < 24:
        print(f"  Prophet: menos de 24 meses de train, la estacionalidad anual puede no ajustar bien.")

    model.fit(df_p)
    future   = pd.DataFrame({'ds': y_test.index})
    forecast = model.predict(future)
    pred     = forecast['yhat'].values[:len(y_test)]
    tiempo   = time.time() - start
    mae  = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2   = r2_score(y_test, pred) if len(y_test) > 1 else float('nan')
    mape = np.mean(np.abs((y_test.values - pred) / np.where(y_test.values == 0, 1, y_test.values))) * 100
    return {
        'modelo': 'Prophet',
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo_entrenamiento': tiempo,
        'predicciones': pred.tolist(), 'real': y_test.tolist()
    }

for target in TARGETS:
    y_train = train[target]
    y_test  = test[target]
    res = entrenar_evaluar_prophet(y_train, y_test)
    res['target'] = target
    resultados.append(res)
    print(f"Prophet - {target}: MAE={res['MAE']:.2f}, R2={res['R2']:.4f}, Tiempo={res['tiempo_entrenamiento']:.2f}s")

00:08:54 - cmdstanpy - INFO - Chain [1] start processing
00:08:54 - cmdstanpy - INFO - Chain [1] done processing
00:08:54 - cmdstanpy - INFO - Chain [1] start processing


Prophet - servicios_totales: MAE=5.75, R2=-0.4284, Tiempo=0.44s


00:08:54 - cmdstanpy - INFO - Chain [1] done processing


Prophet - monto_total: MAE=101537.88, R2=-0.8646, Tiempo=0.30s


Celda 5: XGBoost con Lags

In [17]:
def entrenar_evaluar_xgboost(train_df, test_df, target):
    start = time.time()
    df_all = pd.concat([train_df[[target]], test_df[[target]]])

    lags = [1, 2, 3, 6] if len(train_df) >= 9 else [1, 2, 3]
    for lag in lags:
        df_all[f'lag_{lag}'] = df_all[target].shift(lag)
    df_all['mes'] = df_all.index.month

    feature_cols = [f'lag_{l}' for l in lags] + ['mes']
    df_all = df_all.dropna()

    mask_train = df_all.index.isin(train_df.index)
    X_train_xgb = df_all[mask_train][feature_cols]
    y_train_xgb = df_all[mask_train][target]
    X_test_xgb  = df_all[~mask_train][feature_cols]
    y_test_xgb  = df_all[~mask_train][target]

    if len(X_train_xgb) < 2 or len(X_test_xgb) == 0:
        print(f"  XGBoost - {target}: datos insuficientes tras crear lags.")
        return None

    model = XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42, verbosity=0)
    model.fit(X_train_xgb, y_train_xgb)
    pred   = model.predict(X_test_xgb)
    tiempo = time.time() - start

    mae  = mean_absolute_error(y_test_xgb, pred)
    rmse = np.sqrt(mean_squared_error(y_test_xgb, pred))
    r2   = r2_score(y_test_xgb, pred) if len(y_test_xgb) > 1 else 0.0
    mape = np.mean(np.abs((y_test_xgb.values - pred) / np.where(y_test_xgb.values == 0, 1, y_test_xgb.values))) * 100

    return {
        'modelo': 'XGBoost', 'target': target,
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo_entrenamiento': tiempo,
        'predicciones': pred.tolist(), 'real': y_test_xgb.tolist()
    }

for target in TARGETS:
    res = entrenar_evaluar_xgboost(train, test, target)
    if res:
        resultados.append(res)
        print(f"XGBoost - {target}: MAE={res['MAE']:.2f}, R2={res['R2']:.4f}, Tiempo={res['tiempo_entrenamiento']:.2f}s")

XGBoost - servicios_totales: MAE=4.89, R2=-0.0432, Tiempo=0.11s
XGBoost - monto_total: MAE=50955.61, R2=-0.0960, Tiempo=0.11s


Celda 6: LightGBM con Lags

In [18]:
def entrenar_evaluar_lgbm(train_df, test_df, target):
    start = time.time()

    df_all = pd.concat([train_df[[target]], test_df[[target]]])

    lags = [1, 2, 3, 6] if len(train_df) >= 9 else [1, 2, 3]
    for lag in lags:
        df_all[f'lag_{lag}'] = df_all[target].shift(lag)

    df_all['mes']          = df_all.index.month
    df_all['rolling_mean'] = df_all[target].shift(1).rolling(3, min_periods=1).mean()
    df_all['rolling_std']  = df_all[target].shift(1).rolling(3, min_periods=1).std().fillna(0)

    feature_cols = [f'lag_{l}' for l in lags] + ['mes', 'rolling_mean', 'rolling_std']
    df_all = df_all.dropna()

    mask_train  = df_all.index.isin(train_df.index)
    X_train_lgb = df_all[mask_train][feature_cols]
    y_train_lgb = df_all[mask_train][target]
    X_test_lgb  = df_all[~mask_train][feature_cols]
    y_test_lgb  = df_all[~mask_train][target]

    if len(X_train_lgb) < 2 or len(X_test_lgb) == 0:
        print(f"  LightGBM - {target}: datos insuficientes tras crear lags.")
        return None

    model = LGBMRegressor(
        num_leaves=7,
        n_estimators=200,
        learning_rate=0.03,
        min_child_samples=1,  
        min_data_in_leaf=1,     
        random_state=42,
        verbose=-1
    )
    model.fit(X_train_lgb, y_train_lgb)
    pred   = model.predict(X_test_lgb)
    tiempo = time.time() - start

    mae  = mean_absolute_error(y_test_lgb, pred)
    rmse = np.sqrt(mean_squared_error(y_test_lgb, pred))
    r2   = r2_score(y_test_lgb, pred) if len(y_test_lgb) > 1 else 0.0
    mape = np.mean(np.abs((y_test_lgb.values - pred) / np.where(y_test_lgb.values == 0, 1, y_test_lgb.values))) * 100

    return {
        'modelo': 'LightGBM', 'target': target,
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo_entrenamiento': tiempo,
        'predicciones': pred.tolist(), 'real': y_test_lgb.tolist()
    }

for target in TARGETS:
    res = entrenar_evaluar_lgbm(train, test, target)
    if res:
        resultados.append(res)
        print(f"LightGBM - {target}: MAE={res['MAE']:.2f}, R2={res['R2']:.4f}, Tiempo={res['tiempo_entrenamiento']:.2f}s")

LightGBM - servicios_totales: MAE=4.06, R2=0.2492, Tiempo=0.07s
LightGBM - monto_total: MAE=119793.39, R2=-7.0681, Tiempo=0.05s


Celda 7: LSTM

In [19]:
def entrenar_evaluar_lstm(y_train, y_test, window_size=2):
    start = time.time()
    scaler = MinMaxScaler()
    y_train_scaled = scaler.fit_transform(y_train.values.reshape(-1, 1))

    if len(y_train_scaled) <= window_size:
        print(f"  LSTM: train demasiado corto para window_size={window_size}.")
        return None

    X_lst, y_lst = [], []
    for i in range(window_size, len(y_train_scaled)):
        X_lst.append(y_train_scaled[i - window_size:i, 0])
        y_lst.append(y_train_scaled[i, 0])
    X_lst = np.array(X_lst).reshape(-1, window_size, 1)
    y_lst = np.array(y_lst)

    from tensorflow.keras.callbacks import EarlyStopping
    model = Sequential([
        LSTM(64, input_shape=(window_size, 1), return_sequences=True),
        LSTM(32),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')

    es = EarlyStopping(monitor='loss', patience=20, restore_best_weights=True, verbose=0)
    model.fit(X_lst, y_lst, epochs=300, batch_size=max(1, len(X_lst)//4),
              callbacks=[es], verbose=0)

    predicciones = []
    sequence = y_train_scaled[-window_size:].flatten().tolist()

    for _ in range(len(y_test)):
        x_input     = np.array(sequence[-window_size:]).reshape(1, window_size, 1)
        pred_scaled = model.predict(x_input, verbose=0)[0, 0]
        predicciones.append(pred_scaled)
        sequence.append(pred_scaled)

    pred_values = scaler.inverse_transform(
        np.array(predicciones).reshape(-1, 1)
    ).flatten()

    tiempo = time.time() - start
    y_true = y_test.values

    mae  = mean_absolute_error(y_true, pred_values)
    rmse = np.sqrt(mean_squared_error(y_true, pred_values))
    r2   = r2_score(y_true, pred_values) if len(y_true) > 1 else float('nan')
    mape = np.mean(np.abs((y_true - pred_values) / np.where(y_true == 0, 1, y_true))) * 100

    return {
        'modelo': 'LSTM',
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo_entrenamiento': tiempo,
        'predicciones': pred_values.tolist(), 'real': y_true.tolist()
    }

for target in TARGETS:
    y_train = train[target]
    y_test  = test[target]
    res = entrenar_evaluar_lstm(y_train, y_test)
    if res:
        res['target'] = target
        resultados.append(res)
        print(f"LSTM - {target}: MAE={res['MAE']:.2f}, R2={res['R2']:.4f}, Tiempo={res['tiempo_entrenamiento']:.2f}s")

LSTM - servicios_totales: MAE=4.04, R2=0.0918, Tiempo=12.65s
LSTM - monto_total: MAE=94497.12, R2=-0.5029, Tiempo=19.63s


Celda 8: ETS (Exponential Smoothing)

In [20]:
def entrenar_evaluar_ets(y_train, y_test):
    start = time.time()
    best_pred  = None
    best_train_mae = np.inf

    configs = [
        dict(trend='add',  seasonal=None, initialization_method='estimated'),
        dict(trend=None,   seasonal=None, initialization_method='estimated'),
        dict(trend='add',  seasonal=None, initialization_method='heuristic'),
        dict(trend=None,   seasonal=None, initialization_method='heuristic'),
    ]

    for cfg in configs:
        try:
            m   = ExponentialSmoothing(y_train, **cfg)
            fit = m.fit(optimized=True, remove_bias=False)
            train_mae = mean_absolute_error(y_train, fit.fittedvalues)
            if train_mae < best_train_mae:
                best_train_mae = train_mae
                best_pred      = fit.forecast(steps=len(y_test))
                best_cfg       = cfg
        except Exception:
            continue

    if best_pred is None:
        from statsmodels.tsa.holtwinters import SimpleExpSmoothing
        fit      = SimpleExpSmoothing(y_train).fit(smoothing_level=0.3, optimized=False)
        best_pred = fit.forecast(steps=len(y_test))
        best_cfg  = {'fallback': 'SimpleExpSmoothing(alpha=0.3)'}

    print(f"  ETS mejor config: {best_cfg}")
    pred   = best_pred
    tiempo = time.time() - start

    mae  = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2   = r2_score(y_test, pred) if len(y_test) > 1 else float('nan')
    mape = np.mean(np.abs((y_test.values - pred.values) / np.where(y_test.values == 0, 1, y_test.values))) * 100

    return {
        'modelo': 'ETS',
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo_entrenamiento': tiempo,
        'predicciones': pred.tolist(), 'real': y_test.tolist()
    }

for target in TARGETS:
    y_train = train[target]
    y_test  = test[target]
    res = entrenar_evaluar_ets(y_train, y_test)
    res['target'] = target
    resultados.append(res)
    print(f"ETS - {target}: MAE={res['MAE']:.2f}, R2={res['R2']:.4f}, Tiempo={res['tiempo_entrenamiento']:.2f}s")

  ETS mejor config: {'trend': 'add', 'seasonal': None, 'initialization_method': 'estimated'}
ETS - servicios_totales: MAE=5.09, R2=-0.3561, Tiempo=0.05s
  ETS mejor config: {'trend': 'add', 'seasonal': None, 'initialization_method': 'heuristic'}
ETS - monto_total: MAE=66480.48, R2=-0.0155, Tiempo=0.06s


Celda 9: Tabla Comparativa y Visualizaciones

In [21]:
df_comparativa = pd.DataFrame(resultados)
df_comparativa = df_comparativa[['modelo', 'target', 'MAE', 'RMSE', 'R2', 'MAPE', 'tiempo_entrenamiento']]

print("Tabla Comparativa:")
print(df_comparativa.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, metric in enumerate(['MAE', 'RMSE', 'R2']):
    ax = axes[i]
    for target in TARGETS:
        subset = df_comparativa[df_comparativa['target'] == target]
        ax.bar(subset['modelo'] + ' (' + target[:4] + ')', subset[metric], label=target)
    ax.set_title(metric)
    ax.tick_params(axis='x', rotation=45)
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PATH_GRAFICAS, 'comparativa_mae_rmse.png'), dpi=150, bbox_inches='tight')
plt.close()

print(f"Grafico comparativo guardado en: {PATH_GRAFICAS}")

Tabla Comparativa:
  modelo            target           MAE          RMSE         R2        MAPE  tiempo_entrenamiento
  SARIMA servicios_totales      4.471714      6.426825  -0.428726   42.337118              0.114272
  SARIMA       monto_total 160447.259749 331176.557405 -11.137085 2657.260736              0.057007
 Prophet servicios_totales      5.747336      6.426110  -0.428408  109.221000              0.438675
 Prophet       monto_total 101537.876716 129807.463793  -0.864638  970.131234              0.303816
 XGBoost servicios_totales      4.892906      5.491611  -0.043171   58.539086              0.114059
 XGBoost       monto_total  50955.605469  99520.277813  -0.096020   77.545283              0.114688
LightGBM servicios_totales      4.057635      4.658839   0.249222   50.526685              0.074795
LightGBM       monto_total 119793.393713 270015.900342  -7.068149 2515.130985              0.048432
    LSTM servicios_totales      4.036477      5.123966   0.091827   78.550293    

Celda 10: Visualizaciones por Modelo

In [22]:
import os
import matplotlib.pyplot as plt

modelos_nombres = {
    'SARIMA': 'sarima', 'Prophet': 'prophet', 'XGBoost': 'xgboost',
    'LightGBM': 'lgbm', 'LSTM': 'lstm', 'ETS': 'ets'
}

for target in TARGETS:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    target_results = [r for r in resultados if r['target'] == target]

    for idx, res in enumerate(target_results):
        if idx >= len(axes):
            break
        ax = axes[idx]

        real_vals = res['real']
        pred_vals = res['predicciones']
        x_real = range(len(real_vals))
        x_pred = range(len(pred_vals))

        ax.plot(x_real, real_vals, 'b-o', label='Real', markersize=5)
        ax.plot(x_pred, pred_vals, 'r--x', label='Predicho', markersize=5)
        
        ax.set_title(f"{res['modelo']} - {target}\n" \
                     f"MAE={res['MAE']:.2f}, R2={res['R2']:.4f}", fontsize=10)
        
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    for idx in range(len(target_results), len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle(f"Predicciones por Modelo - {target}", fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    plt.savefig(os.path.join(PATH_GRAFICAS, f'predicciones_{target}.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Gráfico guardado: predicciones_{target}.png")

Gráfico guardado: predicciones_servicios_totales.png
Gráfico guardado: predicciones_monto_total.png


Celda 11: Exportación de Resultados

In [23]:
df_comparativa.to_excel(os.path.join(PATH_OUTPUT_DIR, 'comparativa_modelos.xlsx'), index=False)

predicciones_rows = []
for res in resultados:
    for i, (real, pred) in enumerate(zip(res['real'], res['predicciones'])):
        predicciones_rows.append({
            'modelo': res['modelo'],
            'target': res['target'],
            'punto': i,
            'real': real,
            'predicho': pred,
            'error_absoluto': abs(real - pred)
        })

df_predicciones = pd.DataFrame(predicciones_rows)
df_predicciones.to_excel(os.path.join(PATH_OUTPUT_DIR, 'predicciones_todos_modelos.xlsx'), index=False)

print(f"Archivos exportados en: {PATH_OUTPUT_DIR}")
print("Pipeline de modelos temporales completado.")

Archivos exportados en: C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\resultados
Pipeline de modelos temporales completado.
